# Source Yellow Taxi Trip Data

This notebook ingests NYC Yellow Taxi trip data from the [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) public dataset into a Unity Catalog Delta table.

## What This Notebook Does

1. **Accepts parameters** for target environment, year, and month to control which data file is downloaded.
2. **Optionally resets** the Delta sink table and storage path to allow a clean re-load.
3. **Creates an external Unity Catalog Volume** backed by an Azure Data Lake Storage Gen2 inbound container.
4. **Downloads** the monthly Yellow Taxi `.parquet` file from the TLC CloudFront distribution directly into the Unity Catalog Volume.
5. **Reads, casts, and enriches** the raw parquet with standardized types and audit metadata columns (execution ID, timestamp, source file name/size).
6. **Writes** the transformed data to a partitioned Delta table (`trip_year` / `trip_month`) in the ODS layer using `overwrite` with partition-level replacement so re-runs are idempotent.

## Target Table
`{CATALOG}.{ENVIRONMENT}.yellow_taxi_trips`  
Stored in the ODS Azure Data Lake Storage container, partitioned by `trip_year` and `trip_month`.

## Parameters

The following widgets control notebook execution. They can be set manually when running interactively, or passed programmatically via a Databricks job or workflow.

| Parameter | Default | Description |
|---|---|---|
| `ENVIRONMENT` | _(required)_ | Target environment name. Used to scope the Unity Catalog schema and ADLS paths.  For this demo it is your name all lowercase. |
| `YEAR` | `2020` | Four-digit year of the trip data month to download. |
| `MONTH` | `01` | Two-digit month of the trip data to download (e.g. `01`–`12`). |
| `RESET_SINK` | `False` | When `True`, drops the existing Delta table and removes all ODS data for a clean re-load. **Use with caution in production.** |

In [0]:
# Create Notebook Parameter Widgets
dbutils.widgets.removeAll()
dbutils.widgets.text("ENVIRONMENT", "", "Environment")
dbutils.widgets.text("YEAR", "2020", "Trip Year")
dbutils.widgets.text("MONTH", "01", "Trip Month")
dbutils.widgets.dropdown("RESET_SINK", "False", ["True", "False"], "Reset Sink Location?")

## Imports

## Configuration

Defines source and destination paths based on the supplied parameters.

- **`SOURCE_URL`** — Base CloudFront URL for the TLC parquet files.
- **`INBOUND_PATH`** — ADLS Gen2 path for the raw inbound landing zone (used as the Unity Catalog Volume backing location).
- **`VOLUME_URL`** — Unity Catalog Volume path used to write the downloaded file from the driver node.
- **`ODS_PATH`** — ADLS Gen2 path for the Operational Data Store layer where the Delta table is written.
- **`TARGET_TABLE`** — Fully qualified Unity Catalog table name: `{catalog}.{schema}.yellow_taxi_trips`.

## Reset Sink (Optional)

When `RESET_SINK` is `True`, drops the Delta table from Unity Catalog and recursively deletes all files at the ODS target location. This allows a full re-load of the table from scratch.

> **Warning:** This is a destructive operation. Only enable this in non-production environments or when a clean reload is explicitly required.

## Create Inbound Volume

Ensures the ADLS Gen2 inbound directory exists and registers it as an external Unity Catalog Volume under `{catalog}.{schema}.inbound`. This volume is used as the landing zone for the downloaded parquet file and is idempotent — it will not overwrite an existing volume.

## Download Source File

Streams the monthly Yellow Taxi parquet file from the TLC CloudFront endpoint directly into the Unity Catalog Volume using 8 MB chunks. The download timeout is set to 120 seconds. File size is captured in MB for use as an audit column in the next step.

## Transform and Write to Delta Table

Reads the downloaded parquet into a Spark DataFrame, applies explicit type casts to ensure schema consistency, and appends the following audit/lineage columns:

| Column | Type | Description |
|---|---|---|
| `trip_year` | `int` | Year partition derived from the `YEAR` parameter. |
| `trip_month` | `int` | Month partition derived from the `MONTH` parameter. |
| `source_execution_id` | `string` | UUID generated at runtime to identify this specific notebook run. |
| `source_execution_dt` | `timestamp` | Timestamp of when this notebook run executed. |
| `source_file` | `string` | Name of the source parquet file that was downloaded. |
| `source_file_size` | `int` | Size of the downloaded file in MB. |

The DataFrame is then written to the Delta table partitioned by `trip_year` and `trip_month`. The write uses `replaceWhere` to overwrite only the target year/month partition, making re-runs for the same period safe and idempotent without affecting other partitions.

In [0]:
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT")
YEAR = dbutils.widgets.get("YEAR")
MONTH = dbutils.widgets.get("MONTH")
RESET_SINK = (dbutils.widgets.get("RESET_SINK") == "True")


In [0]:
import requests
import uuid

from pyspark.sql.functions import *

In [0]:
SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DEST_PATH  = "abfss://{CONTAINER}@wbcdaqpseudodlsa.dfs.core.windows.net/{ENVIRONMENT}"

CATALOG_NAME = 'raw_edl_qa_poc'
SCHEMA_NAME = ENVIRONMENT

INBOUND_PATH = DEST_PATH.format(CONTAINER = 'inbound', ENVIRONMENT = ENVIRONMENT)
VOLUME_URL = f'/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/inbound'
ODS_PATH = DEST_PATH.format(CONTAINER = 'ods', ENVIRONMENT = ENVIRONMENT)
TARGET_FILE_NAME = f"yellow_tripdata_{YEAR}-{MONTH}.parquet"
SOURCE = f"{SOURCE_URL}/{TARGET_FILE_NAME}"
TARGET = f"{VOLUME_URL}/{TARGET_FILE_NAME}"

TARGET_LOCATION = ODS_PATH + "/yellow_taxi_trips"
TARGET_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.yellow_taxi_trips" 

In [0]:
if RESET_SINK:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")

    dbutils.fs.rm(TARGET_LOCATION, True)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

In [0]:
# Create Inbound Location
dbutils.fs.mkdirs(INBOUND_PATH)

# Create Volume
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.inbound
  LOCATION '{INBOUND_PATH}'
  COMMENT 'Inbound Folder for {ENVIRONMENT}';
""")

In [0]:
# Download
print(f"Downloading from: {SOURCE}")

response = requests.get(SOURCE, stream=True, timeout=120)
response.raise_for_status()

with open(TARGET, "wb") as f:
    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
        f.write(chunk)

import os
size_mb = os.path.getsize(TARGET) / 1024 / 1024
print(f"Saved to {TARGET} ({size_mb:.1f} MB)")

In [0]:
df = spark.read.parquet(TARGET)

df = df.withColumns({
    "VendorID": col("VendorID").cast("int"),
    "passenger_count": col("passenger_count").cast("int"),
    "RatecodeID": col("RatecodeID").cast("int"),
    "payment_type": col("payment_type").cast("int"),
    "PULocationID": col("PULocationID").cast("int"),
    "DOLocationID": col("DOLocationID").cast("int"),
    "airport_fee": col("airport_fee").cast("double"),
    "trip_month": lit(MONTH).cast("int"),
    "trip_year": lit(YEAR).cast("int"),
    "source_execution_id": lit(str(uuid.uuid4())),
    "source_execution_dt": current_timestamp(),
    "source_file": lit(TARGET_FILE_NAME),
    "source_file_size": lit(size_mb).cast("int")
})

(df.write 
  .format("delta") 
  .partitionBy("trip_year", "trip_month")
  .option("path", TARGET_LOCATION) 
  .option("mergeSchema", "true") 
  .mode("overwrite")   
  .option("replaceWhere", f"trip_year = {int(YEAR)} AND trip_month = {int(MONTH)}")
  .saveAsTable(TARGET_TABLE))